[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# State-Space Models: Kalman → S4 → Mamba

The course only a signal processing society can teach properly. Modern sequence architectures (S4, Mamba) are *literally* this curriculum: state-space models ([Kalman](./Intro_AdFilt_KF.ipynb)), convolution kernels ([Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb)), and discretization ([Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)) — rebranded for deep learning. Four sessions from the linear SSM you already know to a trained sequence model, with every identity verified numerically.

## 1. Pre-requisites

- [Kalman](./Intro_AdFilt_KF.ipynb) & [RNN](./Intro_RNN.ipynb) workshops.
- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) for the architecture being challenged.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Linear SSMs Are Convolutions* (~35 min)
**Goal:** prove (numerically) that an LTI state space = one long FIR filter; why that unlocks parallel training.
**Builds on:** [Kalman](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (HiPPO & discretization).

---

## 2. The Identity Everything Rests On

💡 **Intuition.** A linear time-invariant SSM $h_{t} = \bar{A} h_{t-1} + \bar{B} x_t, \; y_t = C h_t$ can be *unrolled*: $y_t = \sum_{k\ge0} C\bar{A}^{k}\bar{B} \, x_{t-k}$ — a *convolution* with kernel $K_k = C \bar A^k \bar B$. That one identity is the whole trick: **train as a convolution** (parallel, FFT-fast, no backprop-through-time vanishing) and **infer as a recurrence** (constant memory per step, unlike attention's growing KV cache). RNN pain and transformer pain, both dodged — for *linear* state dynamics.

In [ ]:
# ORACLE CHECK: recurrence output == convolution output, elementwise
# path 1: run the recurrence
# path 2: materialize the kernel, convolve

# YOUR CODE HERE


**What just happened.** Two completely different computations — a sequential loop over 200 timesteps carrying a hidden state, and a single convolution against a precomputed 200-tap kernel — produced the same output to **1.8e-15**. That is machine precision, so the identity $y_t = \sum_k C\bar A^k\bar B\,x_{t-k}$ is exact rather than approximate, and the `assert` guarantees it stays that way.

**Why this is the most important cell in the workshop.** The two paths have opposite computational profiles, and the identity lets you pick whichever you need:

- **Training** wants the convolution. Every output is computed in parallel, there is no sequential dependency across time, and the whole thing runs by FFT in $O(T\log T)$. Critically, there is no backpropagation *through time* — the gradient does not thread through 200 sequential multiplications, so the vanishing-gradient pathology that limits the [RNN workshop](./Intro_RNN.ipynb) simply has no path to occur.
- **Inference** wants the recurrence. Generating token by token, you carry a fixed-size state $h$ and pay $O(1)$ per step — against a transformer's KV cache, which grows linearly in context and dominates memory at long sequence lengths.

Getting both from one model is the architectural bet of the whole S4 family, and it rests entirely on this equality.

**Read the kernel plot, because it sets up Session 2.** $K_k = C\bar A^k \bar B$ with diagonal $\bar A$ is a weighted sum of geometric decays $\lambda_i^k$ — the eigenvalues are the system's poles, and each contributes a mode dying at its own rate. Here the $\lambda_i$ were drawn uniformly from $[0.7, 0.98]$, and the kernel is visibly dead within a few dozen taps. An SSM whose kernel has collapsed by $k = 50$ cannot possibly use information from 400 steps ago, whatever its parameter count. That is the vanishing-memory problem in linear, fully visible form, and Session 2 is about fixing it.

**The load-bearing assumption.** This identity holds only because the dynamics are *linear and time-invariant*: the same $\bar A$, $\bar B$, $C$ at every step. Insert any per-step nonlinearity, as an LSTM does, and the unrolling is impossible. Remember this when Session 4 makes the matrices input-dependent — the convolution path is exactly what Mamba gives up, and it is worth knowing now what is being spent.

---
### 🕐 Session 2 of 4 — *HiPPO & Discretization: Why S4's A Matrix Is Special* (~35 min)
**Goal:** see why random A forgets; meet the memory-optimal initialization and the continuous-time view.
**Builds on:** Session 1; [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb). &nbsp; **Feeds into:** Session 3 (training an SSM).

---

## 3. Long Memory Is an Initialization Problem

💡 **Intuition.** Session 1's kernel is a sum of geometric decays $\lambda_i^k$ ([Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb): the poles are the modes!). Random stable $A$ ⇒ all modes die at similar rates ⇒ effective memory of a few dozen steps, the RNN disease in linear form. **HiPPO**'s insight: choose $A$ so the state stores *orthogonal-polynomial coefficients of the input's history* — a principled spread of timescales, kernels with long structured tails. S4 = HiPPO-initialized continuous SSM, discretized ([FoSP2 S2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s $\bar{A} = e^{A\Delta}$, in practice bilinear/ZOH) with a *learnable* step size $\Delta$ — the network literally learns its own sampling rate per channel.

In [ ]:
# Kernel shapes: random-diagonal vs HiPPO-style log-spaced timescales

# YOUR CODE HERE


**What just happened.** Same architecture, same number of modes, same equal weighting — only the *placement of the poles* differs. Random poles drawn from $[0.7, 0.95]$ give a kernel whose last significant tap is at step **47**. Poles spread over decades give one still above 1% of its peak at step **399**, the end of the window.

Read that `399+` carefully: the `+` means the kernel never dropped below the threshold inside the range we computed, so 399 is a *lower bound* on the span, not a measurement of it. The honest statement is "at least eight times longer, and we did not find the end." That distinction matters more than it looks, because the whole claim of this session is about reaching horizons we have not bounded.

**Why the random version fails is arithmetic, not mystery.** A single pole $\lambda$ contributes $\lambda^k$, with a memory horizon of roughly $1/(1-\lambda)$ steps. Drawing 32 poles uniformly from $[0.7, 0.95]$ gives horizons between about 3 and 20 steps — every mode is short, so their sum is short too. You cannot build a long memory by averaging many short ones. The decade-spread version deliberately includes poles like $e^{-10^{-3.5}} \approx 0.99968$, whose horizon is thousands of steps, alongside fast ones that preserve local detail.

**The reframe worth carrying away.** Long memory here is not a matter of capacity or training effort — both models have identical capacity, and neither was trained at all. It is an **initialization** problem: the architecture could always represent a 400-step dependency, but random initialization placed every pole where that is unreachable, and gradient descent starting from a dead kernel has no signal telling it to move poles toward 1. This is why HiPPO is a contribution at all.

**Be clear on what we did and did not implement.** Real HiPPO derives $A$ from an orthogonal-polynomial expansion of the input's history, provably optimal for compressing the past under a chosen measure. Our `np.exp(-np.logspace(-3.5, 0, 32))` is a caricature that keeps only the property doing the work in this demo — timescales spread over decades. It reproduces the *shape* of the benefit, not the theory that motivates the specific matrix.

**And what discretization adds.** In the continuous view the poles come from $\bar A = e^{A\Delta}$, so the step size $\Delta$ sets where they land. Making $\Delta$ learnable per channel means the network chooses its own sampling rate — a learned multi-rate filter bank, in [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s language. Session 3's layer parameterises exactly this, via `log_dt` initialised on `torch.linspace(-4, 0, ...)`: the decade spread, made trainable.

---
### 🕐 Session 3 of 4 — *Train a Diagonal SSM* (~40 min)
**Goal:** build an S4-style layer (diagonal, conv-trained) and beat the LSTM on a long-memory task.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (selectivity & Mamba).

---

## 4. The Layer, Assembled

Our layer (an honest simplification of S4D): per channel, learnable log-timescales $\lambda = e^{-e^{\theta}}$, learnable $C$; compute the kernel, convolve via FFT, add a skip and a nonlinearity. Trained **as a convolution**, verified equal to its recurrence.

In [ ]:
# ORACLE: layer's FFT-conv path == naive recurrence, for one channel

# YOUR CODE HERE


**What just happened.** Session 1's identity, re-checked on the real trainable layer rather than a toy: the FFT-convolution forward pass and a hand-written recurrence over the same parameters agree to **2.4e-6**. The layer we are about to train really does have both computational forms.

**The tolerance moved, and the reason is worth naming.** Session 1 agreed to 1.8e-15; here the `assert` allows 1e-4 and we land at 2.4e-6. Nothing degraded — that is the difference between float64 and float32, compounded by an FFT that accumulates round-off across a length-128 transform. Knowing which tolerance is appropriate for which dtype is a practical skill: asserting 1e-15 on float32 would fail on correct code, and asserting 1e-4 on float64 would pass on broken code.

**What the parameterisation is doing.** The poles are $\lambda = e^{-e^{\theta}}$, a double exponential that maps any real $\theta$ into $(0,1)$. That means gradient descent *cannot* produce an unstable system no matter how large a step it takes — stability is enforced by the parameterisation rather than by clipping or a penalty. Compare with the alternative of learning $\lambda$ directly and projecting it back into range: this is cleaner, has no discontinuity in the gradient, and is a nice example of buying a hard guarantee through a change of variables.

**One detail that is easy to miss.** The FFT uses `n=2*T`, zero-padding to double length. Without it the FFT would compute a *circular* convolution, wrapping the end of the sequence around to contaminate the beginning — the model would appear to see the future. The padding is what makes it a linear convolution, and it is the same overlap-save concern from [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) appearing inside a modern architecture.

With the equivalence confirmed, the next cell trains through the convolution path — parallel across all 400 timesteps, with gradients that never traverse a 400-step sequential chain — while the same weights could be run as an $O(1)$-per-step recurrence at inference time. That is the payoff the workshop has been building toward.

In [ ]:
# The long-memory gauntlet: recall the FIRST token's class after T=400 noise steps

# YOUR CODE HERE


**What just happened.** Remember one token across 400 steps of noise: the SSM reaches **93.4%**, the LSTM **54.7%** against a chance floor of 50%. Both models are small, both trained for 300 steps with the same optimizer and learning rate, on the same batches.

**Why the task is all-or-nothing.** A ±1 cue at $t=0$, then 400 steps of pure noise carrying no information whatsoever, then a binary classification. There is no partial credit and nothing to hill-climb toward: a model that has lost position 0 can do no better than output the class prior. This is why the LSTM sits *at chance* rather than merely underperforming — it is not a slightly worse solution, it is the absence of one.

**The mechanism, which is the actual lesson.** The SSM's gradient reaches $t=0$ through a *convolution* — one hop, no sequential chain, and the kernel tap at lag 400 is a directly-parameterised quantity that Session 2 made sure is non-negligible. The LSTM's gradient must traverse 400 sequential steps, each multiplying by a Jacobian, and the product of 400 such factors either vanishes or explodes long before it reaches the cue. Sessions 1 and 2 are both load-bearing here: Session 1's identity gives the short gradient path, Session 2's timescale spread ensures there is a live kernel tap at lag 400 to carry it.

**Now the honest caveats, because the headline invites overreach.** This is *one synthetic task, one seed, 300 training steps, and no hyperparameter search for either model*. It does not establish that SSMs beat LSTMs in general, and it is not a benchmark result. A carefully tuned LSTM given far longer, or gradient clipping and a better initialisation, would do better than 54.7% — the literature on long-range LSTM training is substantial. What this cell legitimately shows is an **optimisation** difference on a long-range dependency: the SSM finds this solution easily and quickly, and the LSTM does not find it within a comparable budget. That is a real and practically relevant claim, and it is narrower than "SSMs are better."

It is also worth noting what the SSM does *not* do here. 93.4% is not 100%, and the residual errors come from the model having to distinguish the cue from noise of comparable amplitude at a single timestep. The architecture solved the memory problem; the remaining gap is a signal-detection problem, which is a different thing.

**And the limitation that motivates Session 4.** This layer is linear and time-invariant — the same kernel applies to every input. It cannot decide to remember one token and ignore another, because it has no mechanism that depends on content. That works perfectly here, where the important token is always at a fixed position. It fails as soon as *which* token matters depends on what the tokens are.

---
### 🕐 Session 4 of 4 — *Selectivity & Mamba (Frontier Sketch)* (~30 min)
**Goal:** understand what Mamba changes — input-dependent dynamics — and what that costs.
**Builds on:** Session 3.

---

## 5. What Mamba Adds — and What It Breaks

> ℹ️ **Frontier sketch.** This session explains the mechanism and its trade-off; a full efficient selective-scan implementation (Mamba's hardware-aware kernel) is beyond a 40-minute session and is *not* implemented here.

💡 **Intuition.** Everything above is **LTI**: the same kernel for every input — the system cannot *decide* to remember this token and forget that one. Mamba makes $\bar B, \bar C, \Delta$ **functions of the current input** ('selective'): a content-controlled gate on the state, per step. The price is exactly the trade this course's structure predicts: input-dependent dynamics are **time-varying**, so the convolution identity of Session 1 *no longer holds* — no FFT training path. Mamba's contribution is showing the recurrence can still be computed fast on GPUs (parallel associative scan + kernel fusion — the [HW-Accelerated](../Intro_GPU/HW_Accelerated_Computing.ipynb) toolbox earning its keep).

The one-line summary of the whole architecture family:

| | RNN/LSTM | Transformer | S4 (LTI SSM) | Mamba (selective) |
|---|---|---|---|---|
| Train | sequential | parallel | parallel (FFT conv) | parallel (assoc. scan) |
| Infer/step | $O(1)$ | $O(T)$ (KV cache) | $O(1)$ | $O(1)$ |
| Content-dependent routing | gates | **attention** | ✗ | **selection** |
| DSP name | nonlinear IIR | data-adaptive kernel | FIR bank w/ learned poles | time-varying system |

In [ ]:
# The selectivity mechanism in miniature (correct but naive O(T) loop — the SKETCH):
# gate Δ_t = f(x_t) controls how much the state updates on each token
        # the gate sees the current token AND the previous one (a 1-step conv, as in Mamba's
        # local conv before the SSM) — Δ depends on the INPUT: this is the selectivity
# task LTI SSMs cannot do: output the last token that FOLLOWED a '2' marker

# YOUR CODE HERE


**What just happened.** MSE **0.0078** on marker-recall, against **0.0839** for predicting the mean — about 11× better than knowing nothing. The baseline is what makes the number readable: 0.0839 is the target's variance, which is exactly the error you achieve by ignoring the input entirely, so the model has genuinely extracted the token following a marker whose position varies from sample to sample.

**No LTI system can do this, and it is worth being precise about why.** Sessions 1–3 produced a fixed kernel: the output is $\sum_k K_k\,x_{t-k}$, weighting purely by *lag*. But the marker lands anywhere in $[5, T{-}1]$, so the informative token is at a different offset every time. A fixed kernel would have to weight all those offsets, which averages the correct token together with dozens of irrelevant ones. The failure is representational — no amount of training fixes it, because the function being asked for is not in the class.

The gate is what changes that. `dt = torch.sigmoid(self.gate(feats))` makes the write strength a function of the *current and previous token*, so the model can learn "when the previous token was a 2, write hard; otherwise hold." The state update $h \leftarrow (1-\delta_t)h + \delta_t x_t$ then behaves as content-controlled memory rather than a fixed decay. That is selectivity, and it is the whole of Mamba's conceptual step.

**And here is the bill, which Session 1 lets us state exactly.** Because $\delta_t$ depends on the input, the dynamics are **time-varying** — a different $\bar A$ at every step. Session 1's unrolling required a *single* $\bar A$ so that $\bar A^k$ could be a fixed kernel. That assumption is now false, so the convolution identity does not hold, and there is no FFT training path. Look at the code: the loop over `t` is not laziness, it is the honest computation, and the comment saying no conv shortcut *exists* is literally true.

So the trade is exact: content-dependent routing bought at the cost of parallel convolutional training. Mamba's actual contribution is making that affordable anyway — a parallel associative scan with kernel fusion that keeps the state in GPU SRAM, so the sequential-looking recurrence still saturates the hardware. The famous architecture paper is, in large part, a [GPU systems](../Intro_GPU/HW_Accelerated_Computing.ipynb) paper.

**What this toy is not.** It is a 16-dimensional gate on 40-step sequences with an $O(T)$ Python loop, trained on a task designed to need selectivity. It demonstrates the mechanism and its cost; it is not Mamba, and it says nothing about how the idea scales. Treat it as the smallest complete example of the principle — which, for a 30-minute session, is the right thing to have.

## 6. Conclusion

Linear SSM = convolution (verified), long memory = timescale spread (HiPPO's gift), training = FFT, inference = recurrence — and Mamba trades the conv identity for content-selective dynamics computed by scan. You can now read the S4/Mamba papers as *signal processing literature*, because that's what they are.

---
## Where next

- [LLMs from the Ground Up](../Intro_Mach_Learn/LLMs_from_the_Ground_Up.ipynb) — the model family SSMs compete with.
- [Kalman](./Intro_AdFilt_KF.ipynb) / [FoSP2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — the two halves this course glued together.
- [Modern Architectures](../Intro_Mach_Learn/Modern_Architectures.ipynb) — where SSM blocks sit in today's model zoo.